In [1]:
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [3]:
soil = pd.read_csv("../../data/processed/soil_data_cleaned.csv")

crop = pd.read_csv("../../data/processed/crop_production_cleaned.csv")

coords = pd.read_csv("../../data/processed/district_registry_geocoded.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/processed/district_registry_geocoded.csv'

In [ ]:
print("="*60)
print("SOIL DATA")
print("="*60)
print(soil.shape)
display(soil.head())

print("="*60)
print("CROP DATA")
print("="*60)
print(crop.shape)
display(crop.head())

print("="*60)
print("COORDINATE DATA")
print("="*60)
print(coords.shape)
display(coords.head())

SOIL DATA
(10853209, 14)


,id,year,state_name,state_code,district_name,district_code,block_name,block_code,village_name,village_code,nutrient_type,nutrient_name,nutrient_level,value
0,3107023,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Copper,Deficient,0
1,3107024,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Copper,Sufficient,85
2,3107025,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Iron,Deficient,0
3,3107026,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Iron,Sufficient,85
4,3107027,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Manganese,Deficient,2


CROP DATA
(575879, 8)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167


COORDINATE DATA
(885, 5)


,state_name,district_name,_merge,Latitude,Longitude
0,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS,Only in Crop,NaN,NaN
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,Matched,NaN,NaN
2,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,Matched,NaN,NaN
3,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMANS,Matched,NaN,NaN
4,ANDHRA PRADESH,ADILABAD,Only in Crop,NaN,NaN


In [ ]:
print("SOIL COLUMNS")
print(soil.columns.tolist())

print()

print("CROP COLUMNS")
print(crop.columns.tolist())

print()

print("COORDINATE COLUMNS")
print(coords.columns.tolist())

SOIL COLUMNS
['id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'block_name', 'block_code', 'village_name', 'village_code', 'nutrient_type', 'nutrient_name', 'nutrient_level', 'value']

CROP COLUMNS
['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production', 'yield']

COORDINATE COLUMNS
['state_name', 'district_name', '_merge', 'Latitude', 'Longitude']


In [ ]:
print("SOIL")

display(soil.isnull().sum())

print()

print("CROP")

display(crop.isnull().sum())

print()

print("COORDINATES")

display(coords.isnull().sum())

SOIL


id                0
year              0
state_name        0
state_code        0
district_name     0
                 ..
village_code      0
nutrient_type     0
nutrient_name     0
nutrient_level    0
value             0
Length: 14, dtype: int64


CROP


State_Name       0
District_Name    0
Crop_Year        0
Season           0
Crop             0
Area             0
Production       0
yield            0
dtype: int64


COORDINATES


state_name         0
district_name      0
_merge             0
Latitude         704
Longitude        704
dtype: int64

In [ ]:
soil["nutrient_name"].value_counts()

nutrient_name
Organic Carbon             1122793
Phosphorus                 1122778
Potassium                  1122777
Nitrogen                   1122703
Soil Ph                    1122664
                            ...   
Iron                        748501
Sulphur                     748491
Copper                      748490
Boron                       748479
Electrical Conductivity     748475
Name: count, Length: 12, dtype: int64

In [ ]:
soil["value"].describe()

count    1.085321e+07
mean     1.409020e+01
std      4.490956e+01
min      0.000000e+00
25%      0.000000e+00
50%      2.000000e+00
75%      1.100000e+01
max      1.020500e+04
Name: value, dtype: float64

In [ ]:
# Keep only the required columns
soil = soil[
    [
        "state_name",
        "district_name",
        "nutrient_name",
        "value"
    ]
].copy()

print(soil.shape)
soil.head()

(10853209, 4)


,state_name,district_name,nutrient_name,value
0,Uttarakhand,Udham Singh Nagar,Copper,0
1,Uttarakhand,Udham Singh Nagar,Copper,85
2,Uttarakhand,Udham Singh Nagar,Iron,0
3,Uttarakhand,Udham Singh Nagar,Iron,85
4,Uttarakhand,Udham Singh Nagar,Manganese,2


In [ ]:
soil.isnull().sum()


state_name       0
district_name    0
nutrient_name    0
value            0
dtype: int64

In [ ]:
soil_avg = (
    soil
    .groupby(
        ["state_name", "district_name", "nutrient_name"],
        as_index=False
    )["value"]
    .mean()
)

soil_avg.head()

,state_name,district_name,nutrient_name,value
0,Andaman And Nicobar Islands,Nicobars,Boron,7.070796
1,Andaman And Nicobar Islands,Nicobars,Copper,7.070796
2,Andaman And Nicobar Islands,Nicobars,Electrical Conductivity,7.008772
3,Andaman And Nicobar Islands,Nicobars,Iron,7.070796
4,Andaman And Nicobar Islands,Nicobars,Manganese,7.070796


In [ ]:
soil_profile = (
    soil_avg
    .pivot(
        index=["state_name", "district_name"],
        columns="nutrient_name",
        values="value"
    )
    .reset_index()
)

soil_profile.head()

nutrient_name,state_name,district_name,Boron,Copper,Electrical Conductivity,Iron,Manganese,Nitrogen,Organic Carbon,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc
0,Andaman And Nicobar Islands,Nicobars,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
1,Andaman And Nicobar Islands,North And Middle Andaman,30.027523,30.022936,30.022936,30.027523,30.027523,20.018349,20.018349,20.018349,20.018349,20.018349,30.027523,30.027523
2,Andaman And Nicobar Islands,South Andamans,16.793814,16.793814,16.793814,16.793814,16.793814,11.195876,11.192440,11.195876,11.195876,11.195876,16.793814,16.793814
3,Andhra Pradesh,Alluri Sitharama Raju,4.039506,4.042222,3.995309,4.039506,4.043457,2.685761,2.670453,2.688889,2.690864,2.645267,4.039506,4.035062
4,Andhra Pradesh,Anakapalli,13.856935,13.548012,13.852085,13.529098,13.508729,9.231490,9.234400,9.246363,9.238927,9.229227,13.859845,13.441319


In [ ]:
print("District Soil Profile Shape:")
print(soil_profile.shape)

print()

print("Columns:")
print(soil_profile.columns.tolist())

District Soil Profile Shape:
(738, 14)

Columns:
['state_name', 'district_name', 'Boron', 'Copper', 'Electrical Conductivity', 'Iron', 'Manganese', 'Nitrogen', 'Organic Carbon', 'Phosphorus', 'Potassium', 'Soil Ph', 'Sulphur', 'Zinc']


In [ ]:
soil_profile.to_csv(
    "../../data/processed/district_soil_profile.csv",
    index=False
)

print("✅ District soil profile saved successfully.")

✅ District soil profile saved successfully.


In [ ]:
# Crop dataset
crop = crop.rename(columns={
    "State_Name": "state_name",
    "District_Name": "district_name",
    "Crop_Year": "year",
    "Crop": "crop",
    "Season": "season",
    "Area": "area",
    "Production": "production",
    "yield": "yield"
})

# Remove extra spaces and convert to uppercase for reliable matching
crop["state_name"] = crop["state_name"].str.strip().str.upper()
crop["district_name"] = crop["district_name"].str.strip().str.upper()

soil_profile["state_name"] = soil_profile["state_name"].str.strip().str.upper()
soil_profile["district_name"] = soil_profile["district_name"].str.strip().str.upper()

print("Crop:", crop.shape)
print("Soil:", soil_profile.shape)

Crop: (575879, 8)
Soil: (738, 14)


In [ ]:
master = crop.merge(
    soil_profile,
    on=["state_name", "district_name"],
    how="left"
)

print("Master Shape:", master.shape)

master.head()

Master Shape: (575879, 20)


,state_name,district_name,year,season,crop,area,production,yield,Boron,Copper,Electrical Conductivity,Iron,Manganese,Nitrogen,Organic Carbon,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796


In [ ]:
print("Missing values after merge:")

master[
    [
        "Nitrogen",
        "Phosphorus",
        "Potassium",
        "Organic Carbon",
        "Soil Ph"
    ]
].isnull().sum()

Missing values after merge:


Nitrogen          92923
Phosphorus        92923
Potassium         92923
Organic Carbon    92923
Soil Ph           92923
dtype: int64

In [ ]:
print("Total crop records:", len(master))

matched = master["Nitrogen"].notna().sum()

print("Matched records:", matched)
print("Coverage:", round((matched / len(master)) * 100, 2), "%")

Total crop records: 575879
Matched records: 482956
Coverage: 83.86 %


In [ ]:
print(sorted(crop["year"].unique()))

[np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]


In [ ]:
unmatched = master[master["Nitrogen"].isna()].copy()

print("Unmatched Records:", len(unmatched))

unmatched_districts = (
    unmatched[["state_name", "district_name"]]
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
)

print("Unique Unmatched Districts:", len(unmatched_districts))

unmatched_districts.head(20)

Unmatched Records: 92923
Unique Unmatched Districts: 180


,state_name,district_name
319544,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS
236444,ANDHRA PRADESH,ADILABAD
201,ANDHRA PRADESH,ANANTAPUR
529671,ANDHRA PRADESH,HYDERABAD
3207,ANDHRA PRADESH,KADAPA
...,...,...
21394,ASSAM,KARIMGANJ
511006,ASSAM,MAJULI
534869,BIHAR,BOKARO
534875,BIHAR,CHATRA


In [ ]:
unmatched = master[master["Nitrogen"].isna()].copy()

print("Unmatched Records:", len(unmatched))

Unmatched Records: 92923


In [ ]:
unmatched_districts = (
    unmatched[["state_name", "district_name"]]
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

print("Unique unmatched districts:", len(unmatched_districts))

unmatched_districts.head(20)

Unique unmatched districts: 180


,state_name,district_name
0,ANDAMAN AND NICOBAR ISLANDS,ANDAMAN AND NICOBAR ISLANDS
1,ANDHRA PRADESH,ADILABAD
2,ANDHRA PRADESH,ANANTAPUR
3,ANDHRA PRADESH,HYDERABAD
4,ANDHRA PRADESH,KADAPA
...,...,...
15,ASSAM,KARIMGANJ
16,ASSAM,MAJULI
17,BIHAR,BOKARO
18,BIHAR,CHATRA


In [ ]:
soil_profile[
    soil_profile["district_name"].str.upper() == "ADILABAD"
]

nutrient_name,state_name,district_name,Boron,Copper,Electrical Conductivity,Iron,Manganese,Nitrogen,Organic Carbon,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc
588,TELANGANA,ADILABAD,111.7,50.98,111.96,50.98,50.98,74.64,74.64,74.64,74.626667,74.626667,111.76,50.98


In [ ]:
soil_profile["district_name"].value_counts().head(20)

district_name
BILASPUR                    2
HAMIRPUR                    2
PRATAPGARH                  2
NICOBARS                    1
NORTH AND MIDDLE ANDAMAN    1
                           ..
GUNTUR                      1
KAKINADA                    1
KRISHNA                     1
KURNOOL                     1
NANDYAL                     1
Name: count, Length: 20, dtype: int64

In [ ]:
# Soil profile using only district name
soil_by_district = soil_profile.drop(columns=["state_name"])

# Try matching only the unmatched records
unmatched_recovered = unmatched.drop(
    columns=soil_profile.columns[2:],  # remove empty soil columns
    errors="ignore"
).merge(
    soil_by_district,
    on="district_name",
    how="left"
)

print(unmatched_recovered.shape)

print("Recovered Records:",
      unmatched_recovered["Nitrogen"].notna().sum())

print("Still Unmatched:",
      unmatched_recovered["Nitrogen"].isna().sum())

(92967, 20)
Recovered Records: 9027
Still Unmatched: 83940


In [ ]:
# Find district names that appear exactly once
unique_districts = (
    soil_profile["district_name"]
    .value_counts()
)

unique_districts = unique_districts[unique_districts == 1].index

print("Unique district names:", len(unique_districts))

Unique district names: 732


In [ ]:
# Keep only districts that occur exactly once
soil_unique = soil_profile[
    soil_profile["district_name"].isin(unique_districts)
].copy()

print(soil_unique.shape)

(732, 14)


In [ ]:
# Remove the empty soil columns from unmatched records
soil_columns = [
    "Boron",
    "Copper",
    "Electrical Conductivity",
    "Iron",
    "Manganese",
    "Nitrogen",
    "Organic Carbon",
    "Phosphorus",
    "Potassium",
    "Soil Ph",
    "Sulphur",
    "Zinc"
]

unmatched_clean = unmatched.drop(columns=soil_columns, errors="ignore")

# Recover using only unique district names
recovered = unmatched_clean.merge(
    soil_unique.drop(columns=["state_name"]),
    on="district_name",
    how="left"
)

print("Recovered Shape:", recovered.shape)
print("Recovered Records:", recovered["Nitrogen"].notna().sum())
print("Still Missing:", recovered["Nitrogen"].isna().sum())

Recovered Shape: (92923, 20)
Recovered Records: 8939
Still Missing: 83984


In [ ]:
# Keep only the records that were recovered successfully
recovered_success = recovered[recovered["Nitrogen"].notna()].copy()

# Keep the original matched records
master_matched = master[master["Nitrogen"].notna()].copy()

# Combine them
master_final = pd.concat(
    [master_matched, recovered_success],
    ignore_index=True
)

print("Final Dataset Shape:", master_final.shape)
print("Coverage:", round(len(master_final) / len(crop) * 100, 2), "%")

Final Dataset Shape: (491895, 20)
Coverage: 85.42 %


In [ ]:
master_final.to_csv(
    "../../data/processed/master_soil_crop_v2.csv",
    index=False
)

## Weather


In [ ]:
weather_requests = (
    master_final[
        ["state_name", "district_name", "year"]
    ]
    .drop_duplicates()
    .sort_values(
        ["state_name", "district_name", "year"]
    )
    .reset_index(drop=True)
)

print(weather_requests.shape)

weather_requests.head()

(12280, 3)


,state_name,district_name,year
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2001
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2002
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2003
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2004


In [ ]:
coords = pd.read_csv("../../data/processed/district_registry_geocoded.csv")

print(coords.columns.tolist())

print(coords.head())

['state_name', 'district_name', '_merge', 'Latitude', 'Longitude']
                    state_name                district_name        _merge  \
0  ANDAMAN AND NICOBAR ISLANDS  ANDAMAN AND NICOBAR ISLANDS  Only in Crop   
1  ANDAMAN AND NICOBAR ISLANDS                     NICOBARS       Matched   
2  ANDAMAN AND NICOBAR ISLANDS     NORTH AND MIDDLE ANDAMAN       Matched   
3  ANDAMAN AND NICOBAR ISLANDS               SOUTH ANDAMANS       Matched   
4               ANDHRA PRADESH                     ADILABAD  Only in Crop   

   Latitude  Longitude  
0       NaN        NaN  
1       NaN        NaN  
2       NaN        NaN  
3       NaN        NaN  
4       NaN        NaN  


In [ ]:
print(coords[["Latitude", "Longitude"]].isna().sum())

print()

print("Rows with coordinates:",
      coords.dropna(subset=["Latitude", "Longitude"]).shape[0])

print("Total rows:",
      len(coords))

Latitude     704
Longitude    704
dtype: int64

Rows with coordinates: 181
Total rows: 885


In [ ]:
import os

for file in os.listdir("../../data/processed"):
    print(file)

crop_production_cleaned.csv
district_coordinates_2022.csv
district_coordinates_gadm.csv
district_coordinate_schema.csv
district_coordinate_template.csv
district_crop_database.csv
district_crop_database_final.csv
district_crop_database_standardized.csv
district_manual_mapping.csv
district_matching_report.csv
district_registry.csv
district_registry_final.csv
district_registry_geocoded.csv
district_soil_database.csv
district_soil_profile.csv
district_weather_template.csv
high_confidence_geospatial_matches.csv
high_confidence_mappings.csv
manual_review_geospatial_matches.csv
manual_review_mappings.csv
master_soil_crop_v2.csv
missing_district_coordinates.csv
soil_data_cleaned.csv


In [ ]:
gadm = pd.read_csv("../../data/processed/district_coordinates_gadm.csv")

print(gadm.shape)
print(gadm.columns.tolist())

gadm.head()

(676, 4)
['state_name', 'district_name', 'Latitude', 'Longitude']


,state_name,district_name,Latitude,Longitude
0,Andaman and Nicobar,Nicobar Islands,7.527113,93.598609
1,Andaman and Nicobar,North and Middle Andaman,12.853978,92.872983
2,Andaman and Nicobar,South Andaman,11.529706,92.665395
3,Andhra Pradesh,Anantapur,14.476119,77.570671
4,Andhra Pradesh,Chittoor,13.457116,79.004056


In [ ]:
# Standardize names
gadm["state_name"] = gadm["state_name"].str.strip().str.upper()
gadm["district_name"] = gadm["district_name"].str.strip().str.upper()

# Merge with our master dataset
weather_base = master_final.merge(
    gadm,
    on=["state_name", "district_name"],
    how="left"
)

print("Master Shape:", weather_base.shape)

print("Rows with Coordinates:",
      weather_base.dropna(subset=["Latitude", "Longitude"]).shape[0])

print("Coverage:",
      round(
          weather_base["Latitude"].notna().mean() * 100,
          2
      ),
      "%"
)

Master Shape: (495843, 22)
Rows with Coordinates: 453453
Coverage: 91.45 %


In [ ]:
weather_districts = (
    weather_base[
        ["state_name", "district_name", "Latitude", "Longitude"]
    ]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
# Create one request per district

weather_districts = (
    weather_base[
        [
            "state_name",
            "district_name",
            "Latitude",
            "Longitude"
        ]
    ]
    .dropna(subset=["Latitude", "Longitude"])
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

print("Unique districts:", len(weather_districts))

weather_districts.head()

Unique districts: 533


,state_name,district_name,Latitude,Longitude
0,ANDHRA PRADESH,CHITTOOR,13.457116,79.004056
1,ANDHRA PRADESH,EAST GODAVARI,17.185035,82.000952
2,ANDHRA PRADESH,GUNTUR,16.291796,80.081110
3,ANDHRA PRADESH,KRISHNA,16.551226,80.792587
4,ANDHRA PRADESH,KURNOOL,15.528543,78.000024


In [ ]:
weather_districts.to_csv(
    "../../data/processed/weather_districts.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [ ]:
import requests
import pandas as pd
from pathlib import Path

In [ ]:
district = weather_districts.iloc[0]

lat = district["Latitude"]
lon = district["Longitude"]

print("District :", district["district_name"])
print("State    :", district["state_name"])
print("Latitude :", lat)
print("Longitude:", lon)

District : CHITTOOR
State    : ANDHRA PRADESH
Latitude : 13.457116445884491
Longitude: 79.00405571438709


In [ ]:
url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    f"?parameters=T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN"
    f"&community=AG"
    f"&latitude={lat}"
    f"&longitude={lon}"
    f"&start=19970101"
    f"&end=20201231"
    f"&format=JSON"
)

response = requests.get(url, timeout=60)

print("Status Code:", response.status_code)

Status Code: 200


In [ ]:
data = response.json()

print(data.keys())

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [ ]:
print(data["properties"].keys())

dict_keys(['parameter'])


In [ ]:
print(data["properties"]["parameter"].keys())

dict_keys(['T2M', 'PRECTOTCORR', 'RH2M', 'WS2M', 'ALLSKY_SFC_SW_DWN'])


In [ ]:
t2m = data["properties"]["parameter"]["T2M"]

print(type(t2m))
print("Number of days:", len(t2m))

# Show first 5 entries
list(t2m.items())[:5]

<class 'dict'>
Number of days: 8766


[('19970101', 17.63),
 ('19970102', 18.02),
 ('19970103', 18.98),
 ('19970104', 18.8),
 ('19970105', 17.86)]

In [ ]:
t2m_df = (
    pd.DataFrame(
        t2m.items(),
        columns=["date", "temperature"]
    )
)

t2m_df.head()

,date,temperature
0,19970101,17.63
1,19970102,18.02
2,19970103,18.98
3,19970104,18.80
4,19970105,17.86


In [ ]:
weather_df = pd.DataFrame({
    "date": list(data["properties"]["parameter"]["T2M"].keys()),
    "temperature": list(data["properties"]["parameter"]["T2M"].values()),
    "rainfall": list(data["properties"]["parameter"]["PRECTOTCORR"].values()),
    "humidity": list(data["properties"]["parameter"]["RH2M"].values()),
    "wind_speed": list(data["properties"]["parameter"]["WS2M"].values()),
    "solar_radiation": list(data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"].values()),
})

weather_df.head()

,date,temperature,rainfall,humidity,wind_speed,solar_radiation
0,19970101,17.63,0.0,78.86,2.04,17.96
1,19970102,18.02,0.0,79.88,1.51,18.04
2,19970103,18.98,0.0,78.54,1.04,17.05
3,19970104,18.80,0.0,76.25,1.54,17.44
4,19970105,17.86,0.0,79.53,1.96,17.02


In [ ]:
weather_df["state_name"] = district["state_name"]
weather_df["district_name"] = district["district_name"]

# Reorder columns
weather_df = weather_df[
    [
        "state_name",
        "district_name",
        "date",
        "temperature",
        "rainfall",
        "humidity",
        "wind_speed",
        "solar_radiation"
    ]
]

weather_df.head()

,state_name,district_name,date,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDHRA PRADESH,CHITTOOR,19970101,17.63,0.0,78.86,2.04,17.96
1,ANDHRA PRADESH,CHITTOOR,19970102,18.02,0.0,79.88,1.51,18.04
2,ANDHRA PRADESH,CHITTOOR,19970103,18.98,0.0,78.54,1.04,17.05
3,ANDHRA PRADESH,CHITTOOR,19970104,18.80,0.0,76.25,1.54,17.44
4,ANDHRA PRADESH,CHITTOOR,19970105,17.86,0.0,79.53,1.96,17.02


In [ ]:
weather_df["date"] = pd.to_datetime(
    weather_df["date"],
    format="%Y%m%d"
)

weather_df["year"] = weather_df["date"].dt.year

weather_df.head()

,state_name,district_name,date,temperature,rainfall,humidity,wind_speed,solar_radiation,year
0,ANDHRA PRADESH,CHITTOOR,1997-01-01,17.63,0.0,78.86,2.04,17.96,1997
1,ANDHRA PRADESH,CHITTOOR,1997-01-02,18.02,0.0,79.88,1.51,18.04,1997
2,ANDHRA PRADESH,CHITTOOR,1997-01-03,18.98,0.0,78.54,1.04,17.05,1997
3,ANDHRA PRADESH,CHITTOOR,1997-01-04,18.80,0.0,76.25,1.54,17.44,1997
4,ANDHRA PRADESH,CHITTOOR,1997-01-05,17.86,0.0,79.53,1.96,17.02,1997


In [ ]:
yearly_weather = (
    weather_df.groupby(
        ["state_name", "district_name", "year"],
        as_index=False
    )
    .agg({
        "temperature": "mean",
        "rainfall": "sum",
        "humidity": "mean",
        "wind_speed": "mean",
        "solar_radiation": "mean"
    })
)

yearly_weather.head()

,state_name,district_name,year,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDHRA PRADESH,CHITTOOR,1997,24.919123,852.26,68.333205,2.833452,18.610904
1,ANDHRA PRADESH,CHITTOOR,1998,24.993014,996.30,71.368685,2.713068,18.542027
2,ANDHRA PRADESH,CHITTOOR,1999,24.424521,759.10,67.848301,2.852027,19.792219
3,ANDHRA PRADESH,CHITTOOR,2000,24.577049,871.78,68.637486,2.930956,19.260765
4,ANDHRA PRADESH,CHITTOOR,2001,24.801781,863.65,68.035808,3.005479,19.098192


In [ ]:
print("Shape:", yearly_weather.shape)
print("Years:", yearly_weather["year"].min(), "-", yearly_weather["year"].max())

Shape: (24, 8)
Years: 1997 - 2020


In [ ]:
def download_weather(latitude, longitude):
    """
    Download daily weather data (1997-2020)
    from NASA POWER API.
    """

    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters=T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN"
        f"&community=AG"
        f"&latitude={latitude}"
        f"&longitude={longitude}"
        f"&start=19970101"
        f"&end=20201231"
        f"&format=JSON"
    )

    response = requests.get(url, timeout=60)

    response.raise_for_status()

    return response.json()

In [ ]:
def parse_weather(data, state_name, district_name):
    """
    Convert NASA POWER JSON to a daily weather DataFrame.
    """

    weather_df = pd.DataFrame({
        "date": list(data["properties"]["parameter"]["T2M"].keys()),
        "temperature": list(data["properties"]["parameter"]["T2M"].values()),
        "rainfall": list(data["properties"]["parameter"]["PRECTOTCORR"].values()),
        "humidity": list(data["properties"]["parameter"]["RH2M"].values()),
        "wind_speed": list(data["properties"]["parameter"]["WS2M"].values()),
        "solar_radiation": list(data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"].values()),
    })

    weather_df["state_name"] = state_name
    weather_df["district_name"] = district_name

    weather_df["date"] = pd.to_datetime(
        weather_df["date"],
        format="%Y%m%d"
    )

    weather_df["year"] = weather_df["date"].dt.year

    return weather_df

In [ ]:
def aggregate_weather(weather_df):
    """
    Aggregate daily weather data into yearly weather statistics.
    """

    yearly_weather = (
        weather_df
        .groupby(
            ["state_name", "district_name", "year"],
            as_index=False
        )
        .agg({
            "temperature": "mean",
            "rainfall": "sum",
            "humidity": "mean",
            "wind_speed": "mean",
            "solar_radiation": "mean"
        })
    )

    return yearly_weather

In [ ]:
data = download_weather(lat, lon)

daily_weather = parse_weather(
    data,
    district["state_name"],
    district["district_name"]
)

yearly_weather = aggregate_weather(daily_weather)

print(yearly_weather.shape)

yearly_weather.head()

(24, 8)


,state_name,district_name,year,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDHRA PRADESH,CHITTOOR,1997,24.919123,852.26,68.333205,2.833452,18.610904
1,ANDHRA PRADESH,CHITTOOR,1998,24.993014,996.30,71.368685,2.713068,18.542027
2,ANDHRA PRADESH,CHITTOOR,1999,24.424521,759.10,67.848301,2.852027,19.792219
3,ANDHRA PRADESH,CHITTOOR,2000,24.577049,871.78,68.637486,2.930956,19.260765
4,ANDHRA PRADESH,CHITTOOR,2001,24.801781,863.65,68.035808,3.005479,19.098192


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("../../data/weather")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(OUTPUT_DIR)

..\..\data\weather


In [ ]:
all_weather = []

for _, row in weather_districts.head(3).iterrows():

    print(f"Processing: {row['district_name']}")

    data = download_weather(
        row["Latitude"],
        row["Longitude"]
    )

    daily_weather = parse_weather(
        data,
        row["state_name"],
        row["district_name"]
    )

    yearly_weather = aggregate_weather(daily_weather)

    all_weather.append(yearly_weather)

print("\nCompleted!")

Processing: CHITTOOR
Processing: EAST GODAVARI
Processing: GUNTUR

Completed!


In [ ]:
for _, row in weather_districts.iterrows():

    filename = (
        row["state_name"] + "_" + row["district_name"]
    ).replace(" ", "_").replace("/", "_") + ".csv"

    filepath = OUTPUT_DIR / filename

    if filepath.exists():
        print(f"Skipping: {filename}")
        continue

    try:

        print(f"Processing: {row['district_name']}")

        data = download_weather(
            row["Latitude"],
            row["Longitude"]
        )

        daily_weather = parse_weather(
            data,
            row["state_name"],
            row["district_name"]
        )

        yearly_weather = aggregate_weather(daily_weather)

        yearly_weather.to_csv(
            filepath,
            index=False
        )

        print(f"Saved: {filename}")

    except Exception as e:

        print(f"Failed: {filename}")
        print(e)

print("\nAll districts processed!")

Skipping: ANDHRA_PRADESH_CHITTOOR.csv
Skipping: ANDHRA_PRADESH_EAST_GODAVARI.csv
Skipping: ANDHRA_PRADESH_GUNTUR.csv
Processing: KRISHNA
Saved: ANDHRA_PRADESH_KRISHNA.csv
Processing: KURNOOL
Saved: ANDHRA_PRADESH_KURNOOL.csv
Processing: PRAKASAM
Saved: ANDHRA_PRADESH_PRAKASAM.csv
Processing: SRIKAKULAM
Saved: ANDHRA_PRADESH_SRIKAKULAM.csv
Processing: VIZIANAGARAM
Saved: ANDHRA_PRADESH_VIZIANAGARAM.csv
Processing: WEST GODAVARI
Saved: ANDHRA_PRADESH_WEST_GODAVARI.csv
Processing: ANJAW
Saved: ARUNACHAL_PRADESH_ANJAW.csv
Skipping: ARUNACHAL_PRADESH_ANJAW.csv
Processing: CHANGLANG
Saved: ARUNACHAL_PRADESH_CHANGLANG.csv
Processing: DIBANG VALLEY
Saved: ARUNACHAL_PRADESH_DIBANG_VALLEY.csv
Processing: EAST KAMENG
Saved: ARUNACHAL_PRADESH_EAST_KAMENG.csv
Processing: EAST SIANG
Saved: ARUNACHAL_PRADESH_EAST_SIANG.csv
Skipping: ARUNACHAL_PRADESH_EAST_SIANG.csv
Processing: KURUNG KUMEY
Saved: ARUNACHAL_PRADESH_KURUNG_KUMEY.csv
Processing: LOHIT
Saved: ARUNACHAL_PRADESH_LOHIT.csv
Skipping: ARUNACH

In [ ]:
from pathlib import Path
import pandas as pd

weather_files = list(OUTPUT_DIR.glob("*.csv"))

print("Total files:", len(weather_files))

weather_df = pd.concat(
    [pd.read_csv(file) for file in weather_files],
    ignore_index=True
)

print(weather_df.shape)

weather_df.head()

Total files: 525
(12600, 8)


,state_name,district_name,year,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDHRA PRADESH,CHITTOOR,1997,24.919123,852.26,68.333205,2.833452,18.610904
1,ANDHRA PRADESH,CHITTOOR,1998,24.993014,996.30,71.368685,2.713068,18.542027
2,ANDHRA PRADESH,CHITTOOR,1999,24.424521,759.10,67.848301,2.852027,19.792219
3,ANDHRA PRADESH,CHITTOOR,2000,24.577049,871.78,68.637486,2.930956,19.260765
4,ANDHRA PRADESH,CHITTOOR,2001,24.801781,863.65,68.035808,3.005479,19.098192


In [ ]:
downloaded = {
    f.stem.replace("_", " ")
    for f in weather_files
}

expected = {
    (row["state_name"] + "_" + row["district_name"])
    .replace(" ", "_")
    .replace("/", "_")
    for _, row in weather_districts.iterrows()
}

missing = expected - {f.stem for f in weather_files}

print("Missing districts:", len(missing))

for district in sorted(missing):
    print(district)

Missing districts: 0


In [ ]:
print("Unique district names:", weather_districts["district_name"].nunique())
print("Total districts:", len(weather_districts))

duplicates = weather_districts[
    weather_districts.duplicated("district_name", keep=False)
].sort_values("district_name")

duplicates

Unique district names: 519
Total districts: 533


,state_name,district_name,Latitude,Longitude
9,ARUNACHAL PRADESH,ANJAW,27.872394,96.681600
10,ARUNACHAL PRADESH,ANJAW,28.145617,96.879768
57,BIHAR,AURANGABAD,24.793916,84.404576
288,MAHARASHTRA,AURANGABAD,20.023526,75.276043
462,UTTAR PRADESH,BALRAMPUR,27.487531,82.368817
...,...,...,...,...
20,ARUNACHAL PRADESH,LOWER DIBANG VALLEY,28.100666,95.774795
524,UTTARAKHAND,PITHORAGARH,30.095722,80.339014
525,UTTARAKHAND,PITHORAGARH,30.720469,80.151119
406,RAJASTHAN,PRATAPGARH,24.031484,74.681179


In [ ]:
weather_districts = (
    weather_base[
        ["state_name", "district_name", "Latitude", "Longitude"]
    ]
    .dropna(subset=["Latitude", "Longitude"])
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

# Keep only one coordinate per district
weather_districts = (
    weather_districts
    .drop_duplicates(
        subset=["state_name", "district_name"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Unique districts:", len(weather_districts))

Unique districts: 525


In [ ]:
weather_df.to_csv(
    "../../data/processed/weather_yearly.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [ ]:
# Load yearly weather
weather_yearly = pd.read_csv("../../data/processed/weather_yearly.csv")

# Merge with master dataset
master_dataset = master_final.merge(
    weather_yearly,
    on=["state_name", "district_name", "year"],
    how="left"
)

print("Shape:", master_dataset.shape)

print("\nMissing weather values:")
print(master_dataset["temperature"].isna().sum())

master_dataset.head()

Shape: (491895, 25)

Missing weather values:
42390


,state_name,district_name,year,season,crop,area,production,yield,Boron,Copper,Electrical Conductivity,Iron,Manganese,Nitrogen,Organic Carbon,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796,NaN,NaN,NaN,NaN,NaN
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796,NaN,NaN,NaN,NaN,NaN
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796,NaN,NaN,NaN,NaN,NaN
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796,NaN,NaN,NaN,NaN,NaN
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796,NaN,NaN,NaN,NaN,NaN


In [ ]:
print("Shape:", master_dataset.shape)

print("\nMissing values:\n")
print(master_dataset[
    [
        "Nitrogen",
        "Phosphorus",
        "Potassium",
        "temperature",
        "rainfall",
        "humidity",
        "wind_speed",
        "solar_radiation"
    ]
].isna().sum())

print("\nTotal Missing Rows (Weather):")
print(master_dataset["temperature"].isna().sum())

Shape: (491895, 25)

Missing values:

Nitrogen               0
Phosphorus             0
Potassium              0
temperature        42390
rainfall           42390
humidity           42390
wind_speed         42390
solar_radiation    42390
dtype: int64

Total Missing Rows (Weather):
42390


In [ ]:
master_dataset = master_dataset.dropna(
    subset=[
        "temperature",
        "rainfall",
        "humidity",
        "wind_speed",
        "solar_radiation"
    ]
).reset_index(drop=True)

print(master_dataset.shape)

(449505, 25)


In [ ]:
master_dataset.to_csv(
    "../../data/processed/master_dataset.csv",
    index=False
)

print("Final dataset saved successfully!")

Final dataset saved successfully!
